# Notebook 02 — Temporal Split and Universe Freeze

## Objective

Create a deterministic temporal split and freeze the stock universe using **train-period information only**.

## Fixed temporal design

- Model-development period: all observations on or before **2021-03-20**
- Final unseen test: **2021-03-21** through **2024-09-22**
- Observations after 2024-09-22 are outside the current experiment horizon

## Leakage-control rule

The final unseen test is used only for descriptive coverage reporting and later final evaluation. It is not used to select securities, features, labels, models, hyperparameters, thresholds, or trading rules.

## This notebook performs

1. inventories the canonical prepared files from Notebook 01;
2. validates chronology and unique dates again;
3. creates calendar-based train and unseen-test partitions;
4. builds train-only eligibility statistics;
5. freezes the universe from explicit train-only rules;
6. reports unseen-test coverage only after the universe is frozen;
7. writes one train file and one unseen-test file for every selected symbol;
8. creates audit tables and a reproducibility manifest;
9. runs final integrity checks.

> The earlier 470-symbol result is not reused. The universe is recomputed from the 825 canonical prepared files and the current configuration.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


Python: 3.12.2
pandas: 2.2.3
numpy: 1.26.4


## 1. Locate the repository and import project modules


In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Run this notebook from inside the project."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.data.manifest import build_file_inventory
from src.data.splitting import (
    TemporalBoundaries,
    build_test_coverage_audit,
    build_train_only_audit,
    freeze_universe_from_train_only,
    split_by_calendar,
)
from src.utils.paths import data_paths, repository_result_paths, resolve_data_root
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


## 2. Load configuration and freeze the methodological rules


In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
columns_config = load_yaml(REPOSITORY_ROOT / "configs" / "columns.yaml")
validation_config = load_yaml(REPOSITORY_ROOT / "configs" / "validation.yaml")

DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(REPOSITORY_ROOT, paths_config)

PREPARED_DATA_PATH = DATA_PATHS["prepared_data"]
TRAIN_DATA_PATH = DATA_PATHS["train_data"]
UNSEEN_TEST_DATA_PATH = DATA_PATHS["unseen_test_data"]

DATE_COLUMN = "dEven"
OUTPUT_ENCODING = "utf-8-sig"
CALCULATE_INPUT_HASHES = False

boundaries = TemporalBoundaries(
    train_end=pd.Timestamp(validation_config["temporal_design"]["train_end"]),
    unseen_test_start=pd.Timestamp(
        validation_config["temporal_design"]["unseen_test_start"]
    ),
    unseen_test_end=pd.Timestamp(
        validation_config["temporal_design"]["unseen_test_end"]
    ),
)
boundaries.validate()

universe_criteria = dict(validation_config["universe_freeze"])

assert universe_criteria["selection_scope"] == "train_only"
assert universe_criteria["use_candidate_feature_missingness_for_selection"] is False
assert universe_criteria["use_unseen_test_coverage_for_selection"] is False

print("Data root:", DATA_ROOT)
print("Prepared data:", PREPARED_DATA_PATH)
print("Train output:", TRAIN_DATA_PATH)
print("Unseen-test output:", UNSEEN_TEST_DATA_PATH)
print("Train end:", boundaries.train_end.date())
print("Unseen test:", boundaries.unseen_test_start.date(), "to", boundaries.unseen_test_end.date())
print("Universe selection scope:", universe_criteria["selection_scope"])
print(json.dumps(universe_criteria, indent=2))


Data root: E:\Iran_stock_trade\financial-ml-decision-support
Prepared data: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\prepared
Train output: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\train
Unseen-test output: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\unseen_test
Train end: 2021-03-20
Unseen test: 2021-03-21 to 2024-09-22
Universe selection scope: train_only
{
  "criteria_version": 1,
  "selection_scope": "train_only",
  "minimum_train_observations": 600,
  "maximum_train_end_gap_calendar_days": 45,
  "maximum_train_critical_price_missing_fraction": 0.05,
  "maximum_train_any_price_issue_fraction": 0.05,
  "use_candidate_feature_missingness_for_selection": false,
  "use_unseen_test_coverage_for_selection": false
}


### Universe eligibility rules

A symbol is selected only when all four train-period conditions pass:

1. at least 600 train observations;
2. its last train observation is no more than 45 calendar days before the train cutoff;
3. no more than 5% of train rows have a missing critical price;
4. no more than 5% of train rows have any flagged price issue.

Candidate-feature missingness is reported later but is not used here because feature approval is deferred to Notebook 04. Unseen-test coverage is also not used for selection.


## 3. Inventory Notebook 01 outputs


In [4]:
prepared_files = sorted(PREPARED_DATA_PATH.glob("*_prepared.csv"))

if not prepared_files:
    raise FileNotFoundError(
        f"No prepared files were found in {PREPARED_DATA_PATH}. "
        "Run Notebook 01 first."
    )

prepared_inventory_df = build_file_inventory(
    prepared_files,
    calculate_hashes=CALCULATE_INPUT_HASHES,
)
prepared_inventory_path = (
    RESULT_PATHS["manifests"] / "02_prepared_input_inventory.csv"
)
prepared_inventory_df.to_csv(
    prepared_inventory_path,
    index=False,
    encoding=OUTPUT_ENCODING,
)

prepared_input_hash = stable_object_hash(
    prepared_inventory_df.fillna("").to_dict(orient="records")
)

print("Prepared files found:", len(prepared_files))
print("Prepared inventory:", prepared_inventory_path)
print("Prepared input hash:", prepared_input_hash)


Prepared files found: 825
Prepared inventory: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\02_prepared_input_inventory.csv
Prepared input hash: 7a2fff8ca4bcae8009cb6a319bc3b54548bd1acdfc6d5aa6b5a56ffbe90c111b


## 4. Audit every symbol without selecting from unseen-test information


In [5]:
def symbol_from_prepared_path(path: Path) -> str:
    suffix = "_prepared.csv"
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected prepared-file name: {path.name}")
    return path.name[:-len(suffix)]


def read_prepared_file(path: Path) -> pd.DataFrame:
    dataframe = pd.read_csv(path, low_memory=False)
    if DATE_COLUMN not in dataframe.columns:
        raise KeyError(f"{path.name}: missing {DATE_COLUMN}")
    dataframe[DATE_COLUMN] = pd.to_datetime(
        dataframe[DATE_COLUMN],
        errors="coerce",
    )
    return dataframe


train_audit_rows = []
test_coverage_rows = []
split_summary_rows = []
error_rows = []

run_started = time.perf_counter()

for file_number, file_path in enumerate(prepared_files, start=1):
    symbol = symbol_from_prepared_path(file_path)

    try:
        prepared_df = read_prepared_file(file_path)
        train_df, unseen_test_df, after_test_df = split_by_calendar(
            prepared_df,
            boundaries=boundaries,
            date_column=DATE_COLUMN,
        )

        train_audit_rows.append(
            build_train_only_audit(
                symbol,
                train_df,
                boundaries=boundaries,
                date_column=DATE_COLUMN,
            )
        )
        test_coverage_rows.append(
            build_test_coverage_audit(
                symbol,
                unseen_test_df,
                date_column=DATE_COLUMN,
            )
        )

        overlap_rows = int(
            train_df[DATE_COLUMN].isin(unseen_test_df[DATE_COLUMN]).sum()
        )

        split_summary_rows.append(
            {
                "symbol": symbol,
                "prepared_rows": int(len(prepared_df)),
                "train_rows": int(len(train_df)),
                "unseen_test_rows": int(len(unseen_test_df)),
                "after_test_horizon_rows": int(len(after_test_df)),
                "train_test_overlap_rows": overlap_rows,
                "prepared_first_date": prepared_df[DATE_COLUMN].min(),
                "prepared_last_date": prepared_df[DATE_COLUMN].max(),
                "chronological_order_valid": bool(
                    prepared_df[DATE_COLUMN].is_monotonic_increasing
                ),
                "unique_dates_valid": bool(
                    not prepared_df[DATE_COLUMN].duplicated().any()
                ),
            }
        )

    except Exception as exc:
        error_rows.append(
            {
                "symbol": symbol,
                "file_name": file_path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if file_number == 1 or file_number % 25 == 0 or file_number == len(prepared_files):
        elapsed = time.perf_counter() - run_started
        print(
            f"[{file_number:>4}/{len(prepared_files)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(error_rows)}"
        )

first_pass_elapsed_seconds = time.perf_counter() - run_started
print(f"First pass completed in {first_pass_elapsed_seconds:,.1f} seconds.")


[   1/825] elapsed=0.3s errors=0
[  25/825] elapsed=2.0s errors=0
[  50/825] elapsed=3.8s errors=0
[  75/825] elapsed=6.3s errors=0
[ 100/825] elapsed=8.4s errors=0
[ 125/825] elapsed=10.9s errors=0
[ 150/825] elapsed=13.2s errors=0
[ 175/825] elapsed=15.6s errors=0
[ 200/825] elapsed=18.2s errors=0
[ 225/825] elapsed=20.6s errors=0
[ 250/825] elapsed=23.1s errors=0
[ 275/825] elapsed=25.3s errors=0
[ 300/825] elapsed=27.6s errors=0
[ 325/825] elapsed=30.1s errors=0
[ 350/825] elapsed=32.2s errors=0
[ 375/825] elapsed=34.4s errors=0
[ 400/825] elapsed=36.6s errors=0
[ 425/825] elapsed=38.6s errors=0
[ 450/825] elapsed=40.7s errors=0
[ 475/825] elapsed=42.9s errors=0
[ 500/825] elapsed=45.0s errors=0
[ 525/825] elapsed=47.4s errors=0
[ 550/825] elapsed=49.7s errors=0
[ 575/825] elapsed=52.2s errors=0
[ 600/825] elapsed=54.4s errors=0
[ 625/825] elapsed=56.6s errors=0
[ 650/825] elapsed=58.5s errors=0
[ 675/825] elapsed=61.1s errors=0
[ 700/825] elapsed=63.3s errors=0
[ 725/825] elapsed=

## 5. Freeze the universe from train-only fields


In [6]:
train_audit_df = pd.DataFrame(train_audit_rows)
test_coverage_df = pd.DataFrame(test_coverage_rows)
split_summary_df = pd.DataFrame(split_summary_rows)
split_errors_df = pd.DataFrame(error_rows)

universe_audit_df = freeze_universe_from_train_only(
    train_audit_df,
    criteria=universe_criteria,
)

frozen_universe_df = universe_audit_df.loc[
    universe_audit_df["selected_for_frozen_universe"]
].copy()

excluded_universe_df = universe_audit_df.loc[
    ~universe_audit_df["selected_for_frozen_universe"]
].copy()

frozen_symbols = frozen_universe_df["symbol"].tolist()
frozen_symbol_set = set(frozen_symbols)
universe_hash = stable_object_hash(sorted(frozen_symbols))

print("Prepared symbols audited:", len(train_audit_df))
print("Frozen universe size:", len(frozen_universe_df))
print("Excluded symbols:", len(excluded_universe_df))
print("Universe hash:", universe_hash)
print()
print("Decision reasons:")
print(universe_audit_df["universe_decision_reason"].value_counts(dropna=False))


Prepared symbols audited: 825
Frozen universe size: 499
Excluded symbols: 326
Universe hash: bbc7b630cd69ede44ecdfcc82f15f562343100b1c02893d91d48e9fee39dd837

Decision reasons:
universe_decision_reason
selected                                                                                                                           499
insufficient_train_observations|stale_or_missing_at_train_end|excessive_critical_price_missingness|excessive_train_price_issues    161
insufficient_train_observations                                                                                                    130
insufficient_train_observations|stale_or_missing_at_train_end                                                                       26
stale_or_missing_at_train_end                                                                                                        9
Name: count, dtype: int64


In [7]:
# Descriptive test coverage is attached only after the train-only decision exists.
selected_test_coverage_df = (
    frozen_universe_df[["symbol", "selected_for_frozen_universe"]]
    .merge(test_coverage_df, on="symbol", how="left", validate="one_to_one")
    .sort_values("symbol")
    .reset_index(drop=True)
)

print("Selected symbols with zero unseen-test rows:", int(
    (selected_test_coverage_df["unseen_test_rows"] == 0).sum()
))
print("Selected symbols with unseen-test observations:", int(
    (selected_test_coverage_df["unseen_test_rows"] > 0).sum()
))


Selected symbols with zero unseen-test rows: 0
Selected symbols with unseen-test observations: 499


## 6. Write frozen train and unseen-test files

The stage-owned output directories are cleaned before writing so stale files from an earlier run cannot survive.


In [8]:
def remove_stage_files(directory: Path, pattern: str) -> int:
    removed = 0
    for path in directory.glob(pattern):
        path.unlink()
        removed += 1
    return removed


removed_train_files = remove_stage_files(TRAIN_DATA_PATH, "*_train.csv")
removed_test_files = remove_stage_files(
    UNSEEN_TEST_DATA_PATH,
    "*_unseen_test.csv",
)

print("Stale train files removed:", removed_train_files)
print("Stale unseen-test files removed:", removed_test_files)

write_rows = []
write_errors = []
write_started = time.perf_counter()

selected_prepared_files = [
    path
    for path in prepared_files
    if symbol_from_prepared_path(path) in frozen_symbol_set
]

for file_number, file_path in enumerate(selected_prepared_files, start=1):
    symbol = symbol_from_prepared_path(file_path)

    try:
        prepared_df = read_prepared_file(file_path)
        train_df, unseen_test_df, _ = split_by_calendar(
            prepared_df,
            boundaries=boundaries,
            date_column=DATE_COLUMN,
        )

        train_path = TRAIN_DATA_PATH / f"{symbol}_train.csv"
        unseen_test_path = (
            UNSEEN_TEST_DATA_PATH / f"{symbol}_unseen_test.csv"
        )

        train_df.to_csv(
            train_path,
            index=False,
            encoding=OUTPUT_ENCODING,
            date_format="%Y-%m-%d",
        )
        unseen_test_df.to_csv(
            unseen_test_path,
            index=False,
            encoding=OUTPUT_ENCODING,
            date_format="%Y-%m-%d",
        )

        write_rows.append(
            {
                "symbol": symbol,
                "train_rows_written": int(len(train_df)),
                "unseen_test_rows_written": int(len(unseen_test_df)),
                "train_file": train_path.name,
                "unseen_test_file": unseen_test_path.name,
            }
        )

    except Exception as exc:
        write_errors.append(
            {
                "symbol": symbol,
                "file_name": file_path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 25 == 0
        or file_number == len(selected_prepared_files)
    ):
        elapsed = time.perf_counter() - write_started
        print(
            f"[{file_number:>4}/{len(selected_prepared_files)}] "
            f"elapsed={elapsed:,.1f}s "
            f"write_errors={len(write_errors)}"
        )

write_elapsed_seconds = time.perf_counter() - write_started
write_audit_df = pd.DataFrame(write_rows)
write_errors_df = pd.DataFrame(write_errors)

print(f"Write pass completed in {write_elapsed_seconds:,.1f} seconds.")


Stale train files removed: 0
Stale unseen-test files removed: 0
[   1/499] elapsed=0.2s write_errors=0
[  25/499] elapsed=4.0s write_errors=0
[  50/499] elapsed=7.8s write_errors=0
[  75/499] elapsed=11.3s write_errors=0
[ 100/499] elapsed=15.0s write_errors=0
[ 125/499] elapsed=18.7s write_errors=0
[ 150/499] elapsed=22.6s write_errors=0
[ 175/499] elapsed=26.0s write_errors=0
[ 200/499] elapsed=29.8s write_errors=0
[ 225/499] elapsed=34.1s write_errors=0
[ 250/499] elapsed=37.9s write_errors=0
[ 275/499] elapsed=41.6s write_errors=0
[ 300/499] elapsed=45.1s write_errors=0
[ 325/499] elapsed=48.7s write_errors=0
[ 350/499] elapsed=52.4s write_errors=0
[ 375/499] elapsed=56.0s write_errors=0
[ 400/499] elapsed=59.5s write_errors=0
[ 425/499] elapsed=63.1s write_errors=0
[ 450/499] elapsed=66.6s write_errors=0
[ 475/499] elapsed=70.3s write_errors=0
[ 499/499] elapsed=73.8s write_errors=0
Write pass completed in 73.8 seconds.


## 7. Save audit tables and frozen-universe manifest


In [9]:
audit_tables = {
    "02_temporal_split_summary.csv": split_summary_df,
    "02_train_universe_audit.csv": universe_audit_df,
    "02_unseen_test_coverage_audit.csv": selected_test_coverage_df,
    "02_excluded_symbols.csv": excluded_universe_df,
    "02_split_errors.csv": split_errors_df,
    "02_write_audit.csv": write_audit_df,
    "02_write_errors.csv": write_errors_df,
}

for file_name, audit_df in audit_tables.items():
    output_path = RESULT_PATHS["audits"] / file_name
    audit_df.to_csv(output_path, index=False, encoding=OUTPUT_ENCODING)
    print(f"{file_name}: {len(audit_df):,} rows")

frozen_universe_path = RESULT_PATHS["manifests"] / "02_frozen_universe.csv"
frozen_universe_df.to_csv(
    frozen_universe_path,
    index=False,
    encoding=OUTPUT_ENCODING,
)

print("Frozen universe:", frozen_universe_path)


02_temporal_split_summary.csv: 825 rows
02_train_universe_audit.csv: 825 rows
02_unseen_test_coverage_audit.csv: 499 rows
02_excluded_symbols.csv: 326 rows
02_split_errors.csv: 0 rows
02_write_audit.csv: 499 rows
02_write_errors.csv: 0 rows
Frozen universe: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\02_frozen_universe.csv


In [10]:
configuration_snapshot = {
    "paths": paths_config,
    "columns": columns_config,
    "validation": validation_config,
    "calculate_input_hashes": CALCULATE_INPUT_HASHES,
    "output_encoding": OUTPUT_ENCODING,
}

run_manifest = {
    "notebook": "02_temporal_split_and_universe_freeze.ipynb",
    "stage": "temporal_split_and_universe_freeze",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "prepared_input_hash": prepared_input_hash,
    "configuration_hash": stable_object_hash(configuration_snapshot),
    "universe_hash": universe_hash,
    "prepared_files_found": len(prepared_files),
    "symbols_audited_successfully": len(split_summary_df),
    "symbols_failed_during_audit": len(split_errors_df),
    "frozen_universe_size": len(frozen_universe_df),
    "excluded_symbol_count": len(excluded_universe_df),
    "train_files_written": len(list(TRAIN_DATA_PATH.glob("*_train.csv"))),
    "unseen_test_files_written": len(
        list(UNSEEN_TEST_DATA_PATH.glob("*_unseen_test.csv"))
    ),
    "train_end": str(boundaries.train_end.date()),
    "unseen_test_start": str(boundaries.unseen_test_start.date()),
    "unseen_test_end": str(boundaries.unseen_test_end.date()),
    "universe_selection_scope": universe_criteria["selection_scope"],
    "unseen_test_used_for_universe_selection": False,
    "first_pass_elapsed_seconds": first_pass_elapsed_seconds,
    "write_elapsed_seconds": write_elapsed_seconds,
    "software": software_manifest(),
}

run_manifest_path = (
    RESULT_PATHS["manifests"]
    / "02_temporal_split_and_universe_freeze_manifest.json"
)
run_manifest_path.write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

print("Run manifest:", run_manifest_path)
print(json.dumps(run_manifest, ensure_ascii=False, indent=2, default=str))


Run manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\02_temporal_split_and_universe_freeze_manifest.json
{
  "notebook": "02_temporal_split_and_universe_freeze.ipynb",
  "stage": "temporal_split_and_universe_freeze",
  "git_commit_sha": "93665997468a9e24ab9047d460b76bf40a9547d0",
  "prepared_input_hash": "7a2fff8ca4bcae8009cb6a319bc3b54548bd1acdfc6d5aa6b5a56ffbe90c111b",
  "configuration_hash": "48d8678485436248eb80a2152cdc7099c280a0f84b09f45112914eb4aaa15731",
  "universe_hash": "bbc7b630cd69ede44ecdfcc82f15f562343100b1c02893d91d48e9fee39dd837",
  "prepared_files_found": 825,
  "symbols_audited_successfully": 825,
  "symbols_failed_during_audit": 0,
  "frozen_universe_size": 499,
  "excluded_symbol_count": 326,
  "train_files_written": 499,
  "unseen_test_files_written": 499,
  "train_end": "2021-03-20",
  "unseen_test_start": "2021-03-21",
  "unseen_test_end": "2024-09-22",
  "universe_selection_scope": "train_only",
  "unseen_test_used_for_universe_selec

## 8. Final integrity checks


In [11]:
train_files = sorted(TRAIN_DATA_PATH.glob("*_train.csv"))
unseen_test_files = sorted(
    UNSEEN_TEST_DATA_PATH.glob("*_unseen_test.csv")
)

train_file_symbols = {
    path.name[:-len("_train.csv")]
    for path in train_files
}
test_file_symbols = {
    path.name[:-len("_unseen_test.csv")]
    for path in unseen_test_files
}

total_overlap_rows = int(
    split_summary_df["train_test_overlap_rows"].sum()
)

assert split_errors_df.empty, (
    "One or more prepared files failed the temporal audit. "
    "Review results/audits/02_split_errors.csv."
)
assert write_errors_df.empty, (
    "One or more selected files failed writing. "
    "Review results/audits/02_write_errors.csv."
)
assert len(split_summary_df) == len(prepared_files)
assert split_summary_df["chronological_order_valid"].all()
assert split_summary_df["unique_dates_valid"].all()
assert total_overlap_rows == 0
assert len(frozen_universe_df) > 0
assert train_file_symbols == frozen_symbol_set
assert test_file_symbols == frozen_symbol_set
assert len(train_files) == len(frozen_universe_df)
assert len(unseen_test_files) == len(frozen_universe_df)
assert universe_criteria["selection_scope"] == "train_only"
assert universe_criteria["use_unseen_test_coverage_for_selection"] is False

assert (
    universe_audit_df.loc[
        universe_audit_df["selected_for_frozen_universe"],
        "train_rows",
    ]
    >= int(universe_criteria["minimum_train_observations"])
).all()

assert (
    universe_audit_df.loc[
        universe_audit_df["selected_for_frozen_universe"],
        "train_end_gap_calendar_days",
    ]
    <= int(universe_criteria["maximum_train_end_gap_calendar_days"])
).all()

print("Notebook 02 integrity checks: PASSED")
print("Prepared symbols audited:", len(split_summary_df))
print("Frozen universe size:", len(frozen_universe_df))
print("Train files written:", len(train_files))
print("Unseen-test files written:", len(unseen_test_files))
print("Train/Test overlap rows:", total_overlap_rows)
print("Universe selection used unseen-test information: False")
print("Next stage: Notebook 03 — Label reconstruction and censoring.")


Notebook 02 integrity checks: PASSED
Prepared symbols audited: 825
Frozen universe size: 499
Train files written: 499
Unseen-test files written: 499
Train/Test overlap rows: 0
Universe selection used unseen-test information: False
Next stage: Notebook 03 — Label reconstruction and censoring.


## Review before freezing this stage

Inspect these files before committing:

- `results/audits/02_train_universe_audit.csv`
- `results/audits/02_excluded_symbols.csv`
- `results/audits/02_unseen_test_coverage_audit.csv`
- `results/audits/02_split_errors.csv`
- `results/manifests/02_frozen_universe.csv`
- `results/manifests/02_temporal_split_and_universe_freeze_manifest.json`

Do not continue to Notebook 03 until the integrity checks pass and the exclusion reasons are reviewed.

## Recommended Git checkpoint

```bash
git add -A
git commit -m "feat: freeze train-only stock universe and temporal split"
git push origin main
git tag -a milestone-02-universe-freeze -m "Freeze leakage-safe temporal split and stock universe"
git push origin milestone-02-universe-freeze
```
